In [1]:
import os
import base64
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
import magic
import open_clip
from PIL import Image
import torch
import faiss
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer, CrossEncoder
import open_clip
from typing import List,Dict
import numpy as np
from word_handler import Chunk, DocMeta, process_docx



# Force Hugging Face to look directly at your D drive directory bypassing the link
os.environ["HF_HOME"] = r"D:\models\huggingface"
os.environ["TORCH_HOME"] = r"D:\models\torch_models"



C:\Users\shahin\AppData\Local\Temp\ipykernel_16052\3066877382.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
load_dotenv()

hf_api_key = os.getenv("HUGGIN_FACE_API")



text_index    = None
text_meta: Dict[int, Chunk] = {}


image_index   = None
image_meta: Dict[int, dict] = {}


doc_meta_store:  Dict[str, DocMeta]        = {}
image_to_chunks: Dict[tuple[str, int], List[Chunk]]      = {}
table_to_chunks: Dict[tuple[str, int], List[Chunk]]      = {}
all_chunks:      List[Chunk] = []

In [3]:
client = OpenAI(
    api_key=hf_api_key,
    base_url="https://router.huggingface.co/v1"
)

model_name = "Qwen/Qwen3-4B-Instruct-2507"
vision_model_name = "Qwen/Qwen3-VL-8B-Instruct"

# Convert local image file to base64 string
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")




In [ ]:
##test#######
response = client.chat.completions.create(
    model= model_name,
    messages= [
        {
            "role":"user", "content":"give me a simple code for rag using langchain"
        }
    ]
)

# print(response.choices[0].message.content)
display(Markdown(response.choices[0].message.content))



local_image_path = r"C:\Users\shahin\Desktop\pics\er.jfif"
base64_image = encode_image_to_base64(local_image_path)
image_data_url = f"data:image/jpeg;base64,{base64_image}"

# Request execution block
response = client.chat.completions.create(
    model=vision_model_name,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this picture in detail."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url  # Passes the parsed base64 data string variable
                    }
                }
            ]
        }
    ]
)

display(Markdown(response.choices[0].message.content))

In [5]:


file_path =  r"F:\university\az e riz\گزارش.docx"

path = Path(file_path)

suffix, mime = None, None

if path.exists():
    suffix = path.suffix
    mime = magic.from_file(str(path), mime=True)


def route_file(mime, suffix, path):
    if mime == "application/pdf":
        #pdf
        handle_pdf(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        #word
        handle_word(path)


    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        handle_excel(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:

        handle_pp(path)

    # if mime.startswith("image/"):
    #     return "Image"
    #
    # if mime.startswith("audio/"):
    #     return "Audio"
    #
    # if mime.startswith("video/"):
    #     return "Video"

    return "invalid"







###########Alternative using model###################
# print(mime, suffix)
# system_prompt = """
# Your role is just to analyse the MIME and the suffix passed to you and detect the file type.
# You must identify if it is Excel, Word document, Powerpoint, Audio, Video, Image or PDF.
# Just return one word.
# If what is provided to you is not valid just return the word: invalid.
# """
#
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {
#             "role": "user",
#             "content": f"MIME: {mime}, suffix: {suffix}"
#         }
#     ]
# )
#
# print(response.choices[0].message.content)

AttributeError: module 'magic' has no attribute 'from_file'

In [17]:
from word_handler import process_docx

indexed_chunks: set = set()    # (doc_id, chunk_index) already in text_index
indexed_images: set = set()    # (doc_id, image_id)    already in image_index
indexed_docs:   set = set()    # doc_ids already fully ingested


def ingest(chunks: List[Chunk], doc_meta: DocMeta,
           img_to_ch: dict, tbl_to_ch: dict):
    doc_meta_store[doc_meta.doc_id] = doc_meta
    all_chunks.extend(chunks)
    for k, v in img_to_ch.items():
        image_to_chunks.setdefault(k, []).extend(v)
    for k, v in tbl_to_ch.items():
        table_to_chunks.setdefault(k, []).extend(v)

def handle_word(path: str):
    chunks, doc_meta, img_to_ch, tbl_to_ch = process_docx(path)

    if doc_meta.doc_id in indexed_docs:
        print(f"⏭️  skipped {doc_meta.doc_id} (already indexed)")
        return

    ingest(chunks, doc_meta, img_to_ch, tbl_to_ch)
    index_chunks(chunks)
    index_images(chunks)
    indexed_docs.add(doc_meta.doc_id)

def handle_excel(path):
    pass

def handle_pp(path):
    pass


def handle_pdf(path):
    pass

In [7]:
def handle_chunk(chunk, doc_meta):


    content = chunk.text

    if chunk.image_refs:
        content+= "\n[Image]\n" + str(list(chunk.image_refs.values()))

    if chunk.table_refs:
        content += "\n[TABLES]\n" + str(list(chunk.table_refs.values()))

    return Document




**Embeddings models**


1. mulitlangual e5-v2 for sentences
2. open-clip for images


In [8]:
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-base")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)
reranker = CrossEncoder("BAAI/bge-reranker-base", trust_remote_code=True)

clip_model.eval()


def embed_text(texts: List[str], is_query: bool = False) -> np.ndarray:

    prefix  = "query: " if is_query else "passage: "
    prefixed = [prefix + t for t in texts]
    vecs = e5_model.encode(prefixed, normalize_embeddings=True)
    return np.array(vecs, dtype="float32")

def embed_image(image_paths: List[str]) -> np.ndarray:
    images = torch.stack([
        clip_preprocess(Image.open(p).convert("RGB"))
        for p in image_paths
    ])                                              # shape (n, 3, 224, 224)

    with torch.no_grad():
        vecs = clip_model.encode_image(images)     # shape (n, 512)

    vecs = vecs /np.linalg.norm(vecs, keepdims=True) # L2 normalise
    return vecs.cpu().numpy().astype("float32")


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

C:\Users\shahin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\models\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

**Faiss Docstores**

In [18]:
def index_chunks(chunks: List[Chunk]):
    """Embed chunk texts and insert into the text FAISS index."""
    global text_index

    # Skip any chunk already indexed (guards against repeated runs)
    new_chunks = [
        c for c in chunks
        if (c.doc_id, c.chunk_index) not in indexed_chunks
    ]
    if not new_chunks:
        return

    texts = [c.text for c in new_chunks]
    vecs  = embed_text(texts, is_query=False)       # (n, 768)

    if text_index is None:
        text_index = faiss.IndexFlatIP(vecs.shape[1])

    base = text_index.ntotal
    text_index.add(vecs)
    for i, chunk in enumerate(new_chunks):
        text_meta[base + i] = chunk
        indexed_chunks.add((chunk.doc_id, chunk.chunk_index))


def index_images(chunks: List[Chunk]):
    """Embed images from image_context chunks and insert into the image FAISS index."""
    global image_index

    paths, records = [], []
    for c in chunks:
        if not c.image_refs:
            continue
        for img_id, path in c.image_refs.items():
            if (c.doc_id, img_id) in indexed_images:
                continue                             # already indexed
            if os.path.exists(path):
                paths.append(path)
                records.append({
                    "image_id":    img_id,
                    "path":        path,
                    "doc_id":      c.doc_id,
                    "chunk_index": c.chunk_index,
                    "section_path": c.section_path,
                })

    if not paths:
        return

    vecs = embed_image(paths)                       # (m, 512)

    if image_index is None:
        image_index = faiss.IndexFlatIP(vecs.shape[1])

    base = image_index.ntotal
    image_index.add(vecs)
    for i, rec in enumerate(records):
        image_meta[base + i] = rec
        indexed_images.add((rec["doc_id"], rec["image_id"]))

**retreival**

In [20]:


def search_image(query: str, k: int = 5) -> List[dict]:
    if image_index is None or image_index.ntotal == 0:
        return []
    q_token = open_clip.tokenize([query])

    with torch.no_grad:
        q_vec = clip_model.encode_text(q_token)
    q_vec = q_vec/np.linalg.norm(q_vec, keepdims=True)
    q_vec = q_vec.cpu().numpy().astype("float32")

    scores, ids = image_index.search(q_vec, k)

    results = []
    for score, fid in zip(scores[0], ids[0]):
        if fid == -1:
            continue
        meta = image_meta[fid]
        results.append({
            "score":        round(float(score), 4),
            "image_id":     meta["image_id"],
            "path":         meta["path"],
            "doc_id":       meta["doc_id"],
            "chunk_index":  meta["chunk_index"],
            "section_path": meta["section_path"],
        })
    return results

def build_context(text_results:List[dict], image_results:List[dict]) ->str:
    parts = []


    for r in text_results:
        path = " > ".join(r["section_path"]) if r["section_path"] else r["doc_id"]
        parts.append(f"[{path}]\n{r['text']}")

    for r in image_results:
          key = (r["doc_id"], r["image_id"])
          chunks  = image_to_chunks.get(key,[])
          if chunks:
              related = chunks[0]
              path=">".join(r["section_path"] if r["section_path"] else r["doc_id"])
              parts.append(f"[{path}| image_{r["image_id"]}]\n context: {related.text}")

    return "\n___\n".join(parts)

def search_text(query: str, k: int = 5, oversample_factor: int = 4) -> List[dict]:
    if text_index is None or text_index.ntotal == 0:
        return []

    q_vec = embed_text([query], is_query=True)  # Pass as a list to match your embedding function

    # Request more candidates up front so the deduplication guard has room to work
    broad_k = min(k * oversample_factor, text_index.ntotal)
    scores, ids = text_index.search(q_vec, broad_k)

    results = []
    seen_texts = set()  # 🌟 Deduplication Guard: Prevents duplicate strings from leaking into your pool

    for score, fid in zip(scores[0], ids[0]):
        if fid == -1:
            continue
        chunk = text_meta[fid]

        # Deduplicate identical text chunks safely
        if chunk.text in seen_texts:
            continue
        seen_texts.add(chunk.text)

        results.append({
            "faiss_score":  round(float(score), 4),  # Rename this so it doesn't collide with the reranker
            "text":         chunk.text,
            "doc_id":       chunk.doc_id,
            "chunk_index":  chunk.chunk_index,
            "chunk_type":   chunk.chunk_type,
            "section_path": chunk.section_path,
            "image_refs":   chunk.image_refs,
            "table_refs":   chunk.table_refs,
        })

    # Return up to our broad candidate pool limit to let the reranker decide
    return results

def rerank(query: str, chunks: List[dict], top_k: int = 5, threshold: float = 0.1) -> List[dict]:
    if not chunks:
        return []

    pairs = [
        (query, chunk["text"])
        for chunk in chunks
    ]

    # Compute actual cross-encoder logits
    rerank_scores = reranker.predict(pairs)

    # Attach the new scores to your dictionary structure
    reranked_results = []
    for chunk, score in zip(chunks, rerank_scores):
        # Update the score element with the cross-encoder's score
        chunk["score"] = round(float(score), 4)
        reranked_results.append(chunk)

    # Sort descending by the newly assigned cross-encoder score
    reranked_results.sort(key=lambda x: x["score"], reverse=True)

    # Hard-gate filtering: Completely discard out-of-domain chunks
    # BAAI/bge-reranker-base drops below 0.0 or 0.1 for totally unrelated data
    filtered_results = [r for r in reranked_results if r["score"] > threshold]

    return filtered_results[:top_k]


In [25]:

if __name__ == "__main__":
    handle_word(r"F:\university\az e riz\گزارش.docx")

    results = search_text("روش‌شناسی تحقیق", k=8)
    print(results)
    # for r in results:
    #     print(f"score={r['faiss_score']} | {r['section_path']}")
    #     print(f"  {r['text'][:120]}\n")

    results = rerank("روش‌شناسی تحقیق", results, top_k=3)
    print("---------------")
    for r in results:
        print(r)
    #context = build_context(results, [])
    #print(context[:600])


⏭️  skipped گزارش.docx (already indexed)
[{'faiss_score': 0.7781, 'text': '5. حال پس از build و  compile  کردن برنامه به پروتئوس رفته و اجزای زیر را در ابتدا قرار میدهیم.: پردازنده atmega64، یک سون سگمت 4 تایی(در آزمایش های بعدی به آن نیاز داریم ، نماینده GND و VDD 6. حال پایه های متناظر در پردازنده و سون سگمت را به هم وصل میکنیم(از پورت c0 تا c6z را به پایه های a تا  g سون سگمت و پورت  های B0 تا B3  را به پایه های 1 تا 4 سون سگمت ) 7. حال فایل .hex را  که از مرحله قبل تولید شده در برنامه قرار داده و نتیجه را مشاهده میکنیم.\n[TABLE_22] \n d|v|b|a\nh|g|f|r\nl|k|j|i\n5|3|2|1', 'doc_id': 'گزارش.docx', 'chunk_index': 5, 'chunk_type': 'table', 'section_path': [], 'image_refs': {}, 'table_refs': {22: 'd|v|b|a\nh|g|f|r\nl|k|j|i\n5|3|2|1'}}, {'faiss_score': 0.7719, 'text': 'به نام خدا\n\nگزارش آزمایش سوم آز ریز پردازنده\n\nشاهین ترابی 40116693\n\nمحمدرضا عابدین 40120623\n\nمحمدصادق حیدری 40117683\n\nنام آزمایش:نمایشگر هفت قطعه ای\n\nمراحل آزمایش:\n\nبخش اول:\n\nنمایش دنباله 0 تا به صورت متوالی